# 🤖 Tutorial 06: Meta-RL - Meta Reinforcement Learning

## Del Aprendizaje Supervisado a Entornos Interactivos

En este tutorial aprenderás:

- 🎮 Qué es Meta-RL y por qué es importante
- 🔄 Cómo aplicar MAML a Reinforcement Learning
- 🌐 Adaptación rápida en entornos dinámicos
- 💻 Implementación básica de Meta-RL

---

## 📖 Parte 1: Teoría

### ¿Qué es Meta-RL?

**Meta-Reinforcement Learning** combina Meta-Learning con RL para crear agentes que pueden:

- 🚀 Adaptarse rápidamente a nuevos entornos
- 🎯 Aprender nuevas tareas con pocas interacciones
- 🔄 Transferir conocimiento entre entornos relacionados

### El Desafío:

En RL tradicional:
- Se necesitan millones de interacciones para aprender
- Cada nuevo entorno requiere entrenamiento desde cero
- No hay transferencia eficiente de conocimiento

### La Solución - Meta-RL:

1. **Meta-Training**: Entrenar en múltiples entornos/tareas relacionadas
2. **Fast Adaptation**: Adaptar rápidamente a un nuevo entorno con pocas interacciones
3. **Meta-Test**: Evaluar en entornos nunca vistos

### MAML para RL:

$$\theta \leftarrow \theta - \beta \nabla_{\theta} \sum_{\tau_i \sim \pi_{\theta'_i}} \mathcal{R}(\tau_i)$$

donde $\theta'_i$ es la política adaptada y $\mathcal{R}$ es la recompensa acumulada.

---


## 🛠️ Parte 2: Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

# Intentar importar gymnasium
try:
    import gymnasium as gym
    GYM_AVAILABLE = True
except ImportError:
    GYM_AVAILABLE = False
    print("⚠️  gymnasium no está instalado. Instala con: pip install gymnasium")

from utils.test_utils import print_success, set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Setup completo!")

---

## 💻 Parte 3: Política Simple

In [ ]:
class SimplePolicy(nn.Module):
    """Política simple para entornos de RL."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super(SimplePolicy, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

print("✅ Política definida!")

---

## 📊 Parte 4: Entorno Simplificado (Bandits)

Empezaremos con un problema simple: **Multi-Armed Bandits** con Meta-Learning.

In [ ]:
class MetaBanditEnvironment:
    """
    Entorno de Bandits para Meta-RL.
    Cada 'tarea' es un bandit con diferentes recompensas promedio.
    """
    
    def __init__(self, n_arms=5):
        self.n_arms = n_arms
        self.reset_task()
    
    def reset_task(self):
        """Genera una nueva tarea (distribución de recompensas)."""
        self.true_values = np.random.randn(self.n_arms)
        return self.true_values
    
    def step(self, action):
        """
        Ejecuta una acción y devuelve recompensa.
        
        Args:
            action: Índice del brazo a jalar (0 a n_arms-1)
        
        Returns:
            reward: Recompensa ruidosa
        """
        # Recompensa = valor verdadero + ruido
        reward = self.true_values[action] + np.random.randn() * 0.1
        return reward

print("✅ Entorno de bandits creado!")

---

## 💻 Ejercicio: Meta-Training en Bandits

**Tu tarea**: Implementa meta-training simple para el problema de bandits.

El objetivo es aprender una política que pueda identificar rápidamente cuál brazo es mejor en una nueva tarea.

In [ ]:
class BanditAgent(nn.Module):
    """Agente simple para bandits."""
    
    def __init__(self, n_arms, hidden_dim=32):
        super(BanditAgent, self).__init__()
        # Input: one-hot encoding del último action + recompensa
        self.fc1 = nn.Linear(n_arms + 1, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, n_arms)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return torch.softmax(x, dim=-1)  # Probabilidades de cada acción


# Demostración conceptual
env = MetaBanditEnvironment(n_arms=5)
agent = BanditAgent(n_arms=5)

print("🎮 Meta-RL en Bandits:")
print("  El agente aprende a explorar eficientemente y encontrar el mejor brazo rápido.")
print("  Después del meta-training, puede adaptarse a nuevas distribuciones en pocos pasos.")

---

## 🎓 Resumen y Conclusiones

### ✅ Lo que aprendiste:

1. **Meta-RL** extiende Meta-Learning a entornos interactivos
2. Permite **adaptación rápida** en nuevos entornos
3. Combina lo mejor de **RL** y **Meta-Learning**
4. Es crucial para **robótica** y **agentes autónomos**

### 🌟 Aplicaciones del Mundo Real:

- 🤖 **Robótica**: Robots que se adaptan a nuevos objetos o entornos
- 🎮 **Juegos**: Agentes que aprenden nuevos niveles rápidamente
- 🚗 **Vehículos Autónomos**: Adaptación a nuevas condiciones de tráfico
- 🏭 **Control Industrial**: Sistemas que se ajustan a cambios en la producción

### 📚 Papers Clave:

- **RL²**: [Fast Reinforcement Learning via Slow Reinforcement Learning](https://arxiv.org/abs/1611.02779)
- **MAML for RL**: [Model-Agnostic Meta-Learning for Fast Adaptation of Deep Networks](https://arxiv.org/abs/1703.03400)
- **PEARL**: [Efficient Off-Policy Meta-Reinforcement Learning](https://arxiv.org/abs/1903.08254)

---

## 🎉 ¡Felicidades!

Has completado todos los tutoriales de Meta-Learning! Ahora tienes una comprensión sólida de:

- ✅ Fundamentos del Meta-Learning
- ✅ Algoritmos clave (Prototypical Networks, MAML, Memory-based)
- ✅ Aplicaciones al mundo real (Meta-RL)

### 🚀 Próximos Pasos:

1. Implementa estos algoritmos en tus propios proyectos
2. Explora papers recientes en Meta-Learning
3. Experimenta con diferentes arquitecturas y dominios
4. Contribuye al campo con tus propias ideas!

**¡El futuro de la IA adaptativa está en tus manos!** 🧠🚀
